# Tema 02: Matplotlib

**Taller:** Análisis y Visualización Interactiva en Python
**Duración estimada de esta sesión:** 60 minutos
**Herramienta principal:** Matplotlib (pyplot + Programación Orientada a Objetos)
**Modalidad de práctica:** Google Colab

---

> 📌 **Nota para el profesor:** esta notebook está diseñada para proyectarse y ejecutarse en vivo. Las secciones marcadas como **Práctica guiada** se resuelven junto con el grupo; las marcadas como **Práctica independiente** las resuelven los participantes en sus propias copias de la notebook (`Archivo → Guardar una copia en Drive`).

## 🎯 Objetivos de aprendizaje

Al finalizar este tema, el participante será capaz de:

- Explicar la arquitectura de Matplotlib (Figure, Axes) y la diferencia entre la interfaz `pyplot` y la interfaz orientada a objetos.
- Construir gráficos de línea, barra, histograma y dispersión a partir de un DataFrame de Pandas.
- Personalizar títulos, etiquetas, leyendas, colores y anotaciones.
- Combinar múltiples gráficos en una sola figura con `subplots`.
- Exportar una figura como imagen (`.png`) lista para un reporte o presentación.

## 🧠 Contenido teórico

### 1. ¿Por qué Matplotlib?

Matplotlib es la librería de visualización más antigua y fundamental del ecosistema científico de Python (2003). Casi todas las demás librerías de este taller (Seaborn, Pandas `.plot()`) están construidas **sobre** Matplotlib o se inspiran en su API. Entenderla bien es la base para personalizar cualquier gráfico, sin importar la herramienta.

Sus gráficos son **estáticos** (una imagen), a diferencia de Plotly/Bokeh que veremos más adelante y que son **interactivos**. Aun así, sigue siendo la opción preferida para:
- Publicaciones científicas y reportes impresos (control total del resultado, alta resolución).
- Gráficos rápidos de exploración de datos.
- Cuando no se necesita interactividad (zoom, hover, filtros).

### 2. Arquitectura: `Figure` y `Axes`

Todo gráfico en Matplotlib tiene dos componentes principales:

- **`Figure`**: el "lienzo" completo — puede contener uno o varios gráficos (subplots).
- **`Axes`**: cada gráfico individual dentro de la figura (a pesar del nombre, **no** es lo mismo que "ejes" — un `Axes` incluye ejes X/Y, título, leyenda, etc.).

Hay dos formas de trabajar:

| Interfaz | Uso típico | Ejemplo |
|---|---|---|
| **`pyplot` (estado implícito)** | Gráficos rápidos, una sola figura | `plt.plot(x, y); plt.title("...")` |
| **Orientada a objetos (recomendada)** | Múltiples subplots, personalización avanzada, código reutilizable | `fig, ax = plt.subplots(); ax.plot(x, y)` |

En este taller usaremos principalmente la interfaz **orientada a objetos** (`fig, ax = plt.subplots()`) porque escala mejor y es la que recomienda la documentación oficial para código de producción.

### 3. Tipos de gráfico que practicaremos

- **Línea (`ax.plot`)**: series temporales — ideal para nuestro dataset de clima mensual.
- **Barra (`ax.bar` / `ax.barh`)**: comparar categorías (ciudades).
- **Histograma (`ax.hist`)**: distribución de una variable numérica (precipitación).
- **Dispersión (`ax.scatter`)**: relación entre dos variables numéricas (temperatura vs. humedad).
- **Subplots (`plt.subplots(nrows, ncols)`)**: varios gráficos relacionados en una sola figura.

### 4. Buenas prácticas de comunicación visual

- Todo gráfico debe tener **título, etiquetas de ejes y, si aplica, leyenda** — un gráfico sin contexto no comunica nada.
- Cuidado con el "chartjunk": colores y elementos decorativos que no aportan información.
- Elegir el tipo de gráfico según la pregunta: ¿evolución en el tiempo? (línea) ¿comparación entre categorías? (barra) ¿distribución? (histograma) ¿relación? (dispersión).

## ⚙️ Configuración del entorno

Cargaremos `tema02_clima_mensual.csv`: temperatura, precipitación y humedad promedio mensual (2023-2024) para 5 ciudades — dataset sintético con fines educativos.

In [ ]:
# === Carga del dataset ===
# Opción 1 (recomendada una vez publicado el repositorio del taller):
# reemplaza <usuario>/<repositorio> por la ruta real de tu repo de GitHub
# y ejecuta esta celda. Usa el botón "Raw" de GitHub para obtener la URL.
GITHUB_RAW_URL = (
    "https://raw.githubusercontent.com/<usuario>/<repositorio>/main/"
    "datasets/tema02_clima_mensual.csv"
)

import pandas as pd

try:
    df = pd.read_csv(GITHUB_RAW_URL)
    print("Datos cargados desde GitHub ✅  ->", df.shape)
except Exception as e:
    print("No se pudo leer desde GitHub todavía (repo no configurado o sin internet).")
    print("Sube manualmente el archivo 'tema02_clima_mensual.csv' cuando se te solicite.")
    try:
        from google.colab import files
        subido = files.upload()  # selecciona tema02_clima_mensual.csv
        df = pd.read_csv(list(subido.keys())[0])
    except ImportError:
        # Fuera de Colab (por ejemplo, ejecución local de prueba):
        df = pd.read_csv("tema02_clima_mensual.csv")

df.head()

## 🧭 Práctica guiada

### Paso 1 · Gráfico de línea — evolución de la temperatura en una ciudad

In [ ]:
import matplotlib.pyplot as plt

cdmx = df[(df["ciudad"] == "CDMX") & (df["anio"] == 2024)].sort_values("mes")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(cdmx["mes"], cdmx["temp_promedio_c"], marker="o", color="#d62728", label="Temp. promedio")
ax.fill_between(cdmx["mes"], cdmx["temp_min_c"], cdmx["temp_max_c"], alpha=0.15, color="#d62728", label="Rango min-max")

ax.set_title("Temperatura mensual en CDMX (2024)")
ax.set_xlabel("Mes")
ax.set_ylabel("Temperatura (°C)")
ax.set_xticks(range(1, 13))
ax.legend()
ax.grid(alpha=0.3)
plt.show()

### Paso 2 · Gráfico de barras — comparación entre ciudades

In [ ]:
precip_2024 = (
    df[df["anio"] == 2024]
    .groupby("ciudad")["precipitacion_mm"]
    .sum()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 4))
barras = ax.bar(precip_2024.index, precip_2024.values, color="#1f77b4")
ax.bar_label(barras, fmt="%.0f")  # etiqueta el valor encima de cada barra

ax.set_title("Precipitación acumulada por ciudad (2024)")
ax.set_xlabel("Ciudad")
ax.set_ylabel("Precipitación (mm)")
plt.show()

### Paso 3 · Histograma — distribución de una variable

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["precipitacion_mm"], bins=20, color="#2ca02c", edgecolor="white")
ax.set_title("Distribución de la precipitación mensual (todas las ciudades, 2023-2024)")
ax.set_xlabel("Precipitación (mm)")
ax.set_ylabel("Frecuencia (número de meses)")
plt.show()

### Paso 4 · Dispersión — relación entre dos variables

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(df["temp_promedio_c"], df["humedad_promedio_pct"],
                 c=df["mes"], cmap="twilight", alpha=0.8)
ax.set_title("Temperatura vs. Humedad, coloreado por mes")
ax.set_xlabel("Temperatura promedio (°C)")
ax.set_ylabel("Humedad promedio (%)")
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("Mes")
plt.show()

### Paso 5 · Subplots — panel comparativo de 2x2

In [ ]:
ciudades = ["CDMX", "Monterrey", "Guadalajara", "Mérida"]

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)

for ax, ciudad in zip(axes.flat, ciudades):
    datos = df[(df["ciudad"] == ciudad) & (df["anio"] == 2024)].sort_values("mes")
    ax.plot(datos["mes"], datos["temp_promedio_c"], color="#ff7f0e")
    ax.set_title(ciudad)
    ax.grid(alpha=0.3)

fig.suptitle("Temperatura promedio mensual 2024 por ciudad", fontsize=14)
fig.supxlabel("Mes")
fig.supylabel("Temperatura (°C)")
fig.tight_layout()
plt.show()

### Paso 6 · Exportar una figura a imagen

Útil para insertar el gráfico en un reporte, presentación o el `README.md` del repositorio del taller.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(precip_2024.index, precip_2024.values, color="#9467bd")
ax.set_title("Precipitación acumulada por ciudad (2024)")
fig.tight_layout()
fig.savefig("precipitacion_por_ciudad.png", dpi=150)
print("Figura guardada como precipitacion_por_ciudad.png")

## ✍️ Práctica independiente

Trabaja siempre sobre el DataFrame `df` ya cargado.

**Ejercicio 1.** Grafica la evolución de `temp_promedio_c` de **Tijuana** durante 2023 y 2024 en la **misma** figura (dos líneas con colores distintos y leyenda).

In [ ]:
# TODO: tu código aquí

**Ejercicio 2.** Crea un gráfico de barras horizontales (`ax.barh`) con la humedad promedio anual (2024) de cada ciudad, ordenado de mayor a menor.

In [ ]:
# TODO: tu código aquí

**Ejercicio 3.** Genera un panel de 1x2 subplots: a la izquierda un histograma de `temp_promedio_c`, a la derecha un histograma de `humedad_promedio_pct`.

In [ ]:
# TODO: tu código aquí

**Ejercicio 4 (reto).** Usando `ax.scatter`, grafica `precipitacion_mm` vs. `temp_promedio_c` diferenciando cada ciudad con un color distinto (pista: itera sobre `df['ciudad'].unique()` y llama `ax.scatter` una vez por ciudad para que la leyenda se genere automáticamente).

In [ ]:
# TODO: tu código aquí

---
### ✅ Soluciones (referencia para el profesor)

In [ ]:
# Ejercicio 1
tij = df[df["ciudad"] == "Tijuana"]
fig, ax = plt.subplots(figsize=(9, 4))
for anio, color in zip([2023, 2024], ["#1f77b4", "#ff7f0e"]):
    datos = tij[tij["anio"] == anio].sort_values("mes")
    ax.plot(datos["mes"], datos["temp_promedio_c"], marker="o", label=str(anio), color=color)
ax.set_title("Temperatura mensual en Tijuana")
ax.set_xlabel("Mes"); ax.set_ylabel("Temperatura (°C)")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

# Ejercicio 2
hum_2024 = df[df["anio"] == 2024].groupby("ciudad")["humedad_promedio_pct"].mean().sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(hum_2024.index, hum_2024.values, color="#17becf")
ax.set_title("Humedad promedio 2024 por ciudad")
plt.show()

# Ejercicio 3
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df["temp_promedio_c"], bins=15, color="#d62728")
axes[0].set_title("Distribución de temperatura")
axes[1].hist(df["humedad_promedio_pct"], bins=15, color="#2ca02c")
axes[1].set_title("Distribución de humedad")
plt.show()

# Ejercicio 4
fig, ax = plt.subplots(figsize=(8, 5))
for ciudad in df["ciudad"].unique():
    sub = df[df["ciudad"] == ciudad]
    ax.scatter(sub["temp_promedio_c"], sub["precipitacion_mm"], label=ciudad, alpha=0.7)
ax.set_xlabel("Temperatura promedio (°C)"); ax.set_ylabel("Precipitación (mm)")
ax.legend()
plt.show()

## 🔎 Cierre y puente al siguiente tema

Matplotlib nos da control total sobre cada elemento del gráfico, pero requiere bastante código para análisis **estadístico** (comparar distribuciones entre grupos, correlaciones, etc.). En el **Tema 03 (Seaborn)** veremos cómo lograr gráficos estadísticos más ricos con mucho menos código, trabajando directamente sobre DataFrames.

## 📚 Recursos adicionales

- [Documentación oficial de Matplotlib](https://matplotlib.org/stable/index.html)
- [Guía rápida oficial (Quick start)](https://matplotlib.org/stable/tutorials/introductory/quick_start.html)
- [Galería de ejemplos oficiales](https://matplotlib.org/stable/gallery/index.html)
- [Cheat sheets oficiales de Matplotlib](https://matplotlib.org/cheatsheets/)
- [Colores nombrados en Matplotlib](https://matplotlib.org/stable/gallery/color/named_colors.html)